# Lab 8: Interactive Image Processing Application
In this lab, you will build an interactive image processing application using OpenCV, Matplotlib, and ipywidgets in a Jupyter Notebook. You will create an interface with a control panel, image display panels, and a histogram panel.

## Setup
First, let's import the necessary libraries and set up our workspace.

In [ ]:
# Import necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Initialize global variables
original_bgr = None
original_rgb = None
processed_rgb = None

# Create output areas
msg_out = widgets.Output()
img_out = widgets.Output()
hist_out = widgets.Output()

def browse_image_callback(change):
    """
    Callback function to browse and load an image
    """
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        for filename in uploaded:
            global original_bgr, original_rgb, processed_rgb
            original_bgr = cv2.imread(filename)
            if original_bgr is not None:
                original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
                processed_rgb = original_rgb.copy()
                show_images(original_rgb, processed_rgb)
                with msg_out:
                    clear_output(wait=True)
                    print(f"Image '{filename}' loaded successfully.")
            else:
                with msg_out:
                    clear_output(wait=True)
                    print("Failed to load image.")


## Helper Functions
Define the functions to display images and histograms.

In [ ]:
def show_images(original_rgb, processed_rgb):
    """
    Display the original and processed images side by side.
    """
    with img_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(original_rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        axes[1].imshow(processed_rgb)
        axes[1].set_title('Processed Image')
        axes[1].axis('off')
        plt.show()

def show_histogram(processed_rgb):
    """
    Plot the histogram of the processed image.
    """
    with hist_out:
        clear_output(wait=True)
        if processed_rgb is not None:
            gray = cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2GRAY)
            plt.figure(figsize=(5, 3))
            plt.hist(gray.ravel(), bins=256, range=[0,256])
            plt.title('Histogram')
            plt.show()
        else:
            print("No processed image available.")

## Controls and Callbacks
Set up the user interface controls and their corresponding callback functions.

In [ ]:
# Define controls
browse_btn = widgets.Button(description='Browse Image')
grayscale_btn = widgets.Button(description='Convert to Grayscale')
brightness_slider = widgets.IntSlider(value=0, min=-100, max=100, description='Brightness')
resize_dropdown = widgets.Dropdown(options=[('100%', 1.0), ('75%', 0.75), ('50%', 0.5)], description='Resize')
rotation_dropdown = widgets.Dropdown(options=[('0°', 0), ('90°', 90), ('180°', 180), ('270°', 270)], description='Rotation')
hist_btn = widgets.Button(description='Plot Histogram')
save_btn = widgets.Button(description='Save Image')

def grayscale_callback(change):
    """
    Convert the original image to grayscale and update the processed image.
    """
    global processed_rgb
    if original_rgb is not None:
        gray = cv2.cvtColor(original_rgb, cv2.COLOR_RGB2GRAY)
        processed_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        show_images(original_rgb, processed_rgb)
    else:
        with msg_out:
            clear_output(wait=True)
            print("No original image available.")

def brightness_callback(change):
    """
    Adjust the brightness of the original image based on the slider value.
    """
    global processed_rgb
    if original_rgb is not None:
        value = change['new']
        processed_rgb = cv2.convertScaleAbs(original_rgb, alpha=1, beta=value)
        show_images(original_rgb, processed_rgb)
    else:
        with msg_out:
            clear_output(wait=True)
            print("No original image available.")

def resize_callback(change):
    """
    Resize the original image based on the dropdown selection.
    """
    global processed_rgb
    if original_rgb is not None:
        scale = change['new']
        new_dimensions = (int(original_rgb.shape[1] * scale), int(original_rgb.shape[0] * scale))
        processed_rgb = cv2.resize(original_rgb, new_dimensions, interpolation=cv2.INTER_AREA)
        show_images(original_rgb, processed_rgb)
        with msg_out:
            clear_output(wait=True)
            print(f"Image resized to: {new_dimensions}")
    else:
        with msg_out:
            clear_output(wait=True)
            print("No original image available.")

def rotation_callback(change):
    """
    Rotate the original image based on the dropdown selection.
    """
    global processed_rgb
    if original_rgb is not None:
        angle = change['new']
        if angle == 90:
            processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_90_CLOCKWISE)
        elif angle == 180:
            processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_180)
        elif angle == 270:
            processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
        else:
            processed_rgb = original_rgb.copy()
        show_images(original_rgb, processed_rgb)
    else:
        with msg_out:
            clear_output(wait=True)
            print("No original image available.")

def histogram_callback(change):
    """
    Plot the histogram of the processed image.
    """
    show_histogram(processed_rgb)

def save_image_callback(change):
    """
    Save the processed image to Google Drive.
    """
    from datetime import datetime
    if processed_rgb is not None:
        filename = f"processed_image_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        processed_bgr = cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite(filename, processed_bgr)
        with msg_out:
            clear_output(wait=True)
            print(f"Processed image saved as: {filename}")
    else:
        with msg_out:
            clear_output(wait=True)
            print("No processed image to save.")

## Layout
Arrange the components in the notebook.

In [ ]:
# Connect controls to their callback functions
browse_btn.on_click(browse_image_callback)
grayscale_btn.on_click(grayscale_callback)
brightness_slider.observe(brightness_callback, names='value')
resize_dropdown.observe(resize_callback, names='value')
rotation_dropdown.observe(rotation_callback, names='value')
hist_btn.on_click(histogram_callback)
save_btn.on_click(save_image_callback)

# Arrange the layout
controls = widgets.VBox([
    browse_btn,
    grayscale_btn,
    brightness_slider,
    resize_dropdown,
    rotation_dropdown,
    hist_btn,
    save_btn
])

image_display = widgets.VBox([img_out, hist_out])
layout = widgets.HBox([controls, image_display])

# Display the message output area and layout
display(msg_out, layout)

with msg_out:
    print("Welcome! Click 'Browse Image' to start.")